# Plate & vehicle detector: T3 training on free Colab

**Before you start**
1. In Google Drive, create a folder `platedet` and upload `platedet_bundle.tar` (made on the Mac with `make colab-bundle`, about 3 GB) into it.
2. Here: *Runtime → Change runtime type → T4 GPU* (free tier). Don't pick a paid option.

**Then run the cells top to bottom.** Training writes checkpoints to `My Drive/platedet/runs/`. If the session disconnects, reconnect and run all cells again: training **resumes** from the last saved epoch.

Uses only free Colab (approved by the owner). No paid services.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Unpack the bundle onto the Colab disk (faster than reading images from Drive).
import os

BUNDLE = "/content/drive/MyDrive/platedet/platedet_bundle.tar"
WORK = "/content/platedet"
RUNS = "/content/drive/MyDrive/platedet/runs"
if not os.path.exists(f"{WORK}/pyproject.toml"):
    os.makedirs(WORK, exist_ok=True)
    !tar -xf {BUNDLE} -C {WORK}
os.makedirs(RUNS, exist_ok=True)
%cd {WORK}
!cat bundle_info.json

In [ ]:
# Colab already has torch, torchvision, opencv, numpy, pillow, pyyaml, tensorboard.
# YOLOX also needs these (licences checked in pyproject.toml, none GPL/AGPL):
# Versions match uv.lock.
!pip install -q loguru==0.8.0 thop==0.1.1.post2209072238 tabulate==0.10.0 pycocotools==2.0.11

## Run 1: 640 px (main model)
Re-running this cell after a disconnect resumes the same run. Expect roughly 4–6 hours in total; it's fine to spread this over several sessions.

In [ ]:
!PYTHONPATH=src python -m platedet.train.train \
    --config configs/train/yolox_tiny_640_v0.yaml \
    --runs-dir {RUNS} --run-id yolox_tiny_640_v0

## Run 2: 416 px (speed/accuracy comparison)
Start this only after run 1 has finished.

In [ ]:
!PYTHONPATH=src python -m platedet.train.train \
    --config configs/train/yolox_tiny_416_v0.yaml \
    --runs-dir {RUNS} --run-id yolox_tiny_416_v0

## When both runs are done
From `My Drive/platedet/runs/<run>/` download these to the Mac's `runs/<run>/` folder:
`config.yaml`, `run_info.json`, `yolox/best_ckpt.pth`, and the `yolox/tensorboard` folder (optional).
You don't need `latest_ckpt.pth` or `last_epoch_ckpt.pth`.

Then tell Claude, which will run `make export RUN=<run>` and `make eval RUN=<run>` locally.

In [ ]:
# Optional: check progress / training curves.
%load_ext tensorboard
%tensorboard --logdir {RUNS}